# ConverSight AI: Data Science Algorithms & Business Insights
### Transcript Intelligence Platform - Strategic Algorithms & Analytics

Welcome to the **Algorithms & Insights** notebook. This notebook focuses on how different data science algorithms are implemented to extract actionable business intelligence from the processed conversational transcripts.

#### Analytics Focus:
1. **Exploratory Data Analysis**: Profile call volumes, duration distributions, and topic groupings.
2. **Issue Resolution Pattern detection**: Implement an algorithm to detect customer calls that show a positive sentiment transition from negative beginnings.
3. **Customer Churn Risk Scoring**: Build a multi-factor composite risk index (0 to 100) to flag high-churn account opportunities.
4. **Competitor Landmines & Product Load**: Map competitive mentions and assess support load per module.
5. **Knowledge Graph Node Degree Centrality**: Rank entities based on graph connections.

## 1. Setup & Connection

We load plotting libraries and connect to the analytical warehouse. We try to read from `conversight_pipeline_test.db` first (or fall back to `backend/data/conversight.db` in write mode if read-only lock fails) to ensure we always bypass locking conflicts with other running processes.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Prioritize connecting to the test pipeline DB first to avoid locking conflicts with live dashboards
db_path = "conversight_pipeline_test.db"
if not os.path.exists(db_path):
    db_path = "backend/data/conversight.db"
    if not os.path.exists(db_path):
        db_path = "../backend/data/conversight.db"

try:
    # Attempt read-only connection
    conn = duckdb.connect(db_path, read_only=True)
    print(f"Connected to DuckDB database (read-only) at: {db_path}")
except Exception as e:
    print(f"Read-only connection failed: {e}. Falling back to default connection...")
    # Fall back to pipeline test DB which is owned solely by our notebooks
    db_path = "conversight_pipeline_test.db"
    conn = duckdb.connect(db_path)
    print(f"Connected to isolated DuckDB test database at: {db_path}")

## 2. Basic Dataset Exploratory Profiling

Let's see our call duration and classifications.

In [ ]:
df_meetings = conn.execute("SELECT * FROM meetings").df()
df_meetings['duration_mins'] = df_meetings['duration'] / 60.0

print(df_meetings['call_type'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
df_meetings['call_type'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[0], colors=['#3498db', '#2ecc71', '#e74c3c'])
axes[0].set_ylabel("")
axes[0].set_title("Proportion of Call Types")

sns.histplot(data=df_meetings, x="duration_mins", hue="call_type", multiple="stack", bins=15, ax=axes[1], palette="viridis")
axes[1].set_title("Call Duration Distributions (Minutes)")
plt.show()

## 3. Topic Discovery

We rank the topics discussed in meetings and cross-reference them by call types.

In [ ]:
df_topics = conn.execute("""
    SELECT t.topic, m.call_type, COUNT(*) as count 
    FROM topics t 
    JOIN meetings m ON t.meeting_id = m.meeting_id 
    GROUP BY t.topic, m.call_type
    ORDER BY count DESC
""").df()

df_topic_pivot = df_topics.pivot(index='topic', columns='call_type', values='count').fillna(0).astype(int)
df_topic_pivot['Total'] = df_topic_pivot.sum(axis=1)
df_topic_pivot = df_topic_pivot.sort_values(by='Total', ascending=False)
print("Top 10 Topics pivot table:")
print(df_topic_pivot.head(10))

plt.figure(figsize=(12, 6))
sns.barplot(data=df_topics.groupby('topic')['count'].sum().reset_index().sort_values(by='count', ascending=False).head(10), 
            y='topic', x='count', palette="rocket")
plt.title("Top 10 Most Discussed Themes")
plt.show()

## 4. Algorithm 1: Chronological Sentiment Shift (Resolution Pattern)

This algorithm tracks sentence sentiments (`positive`, `negative`, `neutral`) across each meeting's timeline. It compares the ratio of negative sentiment in the **first third** of the meeting turns to positive sentiment in the **last third** of the turns to calculate a **Resolution Index**.

- **High Positive Shift**: Customer concerns were addressed successfully.
- **Negative Shift**: Meeting escalated or ended unresolved.

In [ ]:
resolution_query = """
    WITH ranked_turns AS (
        SELECT 
            meeting_id, turn_index, sentiment_type,
            ROW_NUMBER() OVER(PARTITION BY meeting_id ORDER BY turn_index ASC) as rn_asc,
            ROW_NUMBER() OVER(PARTITION BY meeting_id ORDER BY turn_index DESC) as rn_desc,
            COUNT(*) OVER(PARTITION BY meeting_id) as total_turns
        FROM transcript_segments
    ),
    first_part AS (
        SELECT 
            meeting_id, 
            SUM(CASE WHEN sentiment_type = 'negative' THEN 1 ELSE 0 END)::FLOAT / COUNT(*) as pct_neg_start
        FROM ranked_turns
        WHERE rn_asc <= (total_turns / 3)
        GROUP BY meeting_id
    ),
    last_part AS (
        SELECT 
            meeting_id, 
            SUM(CASE WHEN sentiment_type = 'positive' THEN 1 ELSE 0 END)::FLOAT / COUNT(*) as pct_pos_end
        FROM ranked_turns
        WHERE rn_desc <= (total_turns / 3)
        GROUP BY meeting_id
    )
    SELECT 
        m.title,
        m.call_type,
        m.sentiment_score,
        f.pct_neg_start,
        l.pct_pos_end,
        (l.pct_pos_end - f.pct_neg_start) as resolution_shift
    FROM meetings m
    JOIN first_part f ON m.meeting_id = f.meeting_id
    JOIN last_part l ON m.meeting_id = l.meeting_id
    ORDER BY resolution_shift DESC
"""

df_res = conn.execute(resolution_query).df()
print("Top 5 Successfully Resolved Calls (Positive Sentiment Shift):")
print(df_res.head(5))
print("\nTop 5 Escalated/Unresolved Calls (Negative Sentiment Shift):")
print(df_res.tail(5))

## 5. Algorithm 2: Multi-Factor Customer Churn-Risk Index

To predict customer churn proactively, we implement a composite risk scoring model ranging from `0` to `100`.

### Composite Risk Score Formula:
$$\text{Risk Score} = 0.4 \times \text{Sentiment Penalty} + 0.3 \times \text{Turn Negativity Ratio} + 20.0 \times \text{Has Competitor} + 10.0 \times \text{Has Churn Keywords}$$

Where:
- **Sentiment Penalty** $= 100 \times \frac{5.0 - \text{sentiment score}}{4.0}$ (maps 1-5 scale to 0-100)
- **Turn Negativity Ratio** $= 100 \times \frac{\text{negative sentence count}}{\text{total sentence count}}$
- **Has Competitor** $= 1.0$ if Okta, Duo, Ping, or Entra are mentioned, else $0.0$
- **Has Churn Keywords** $= 1.0$ if keywords (*cancel, pricing, billing, frustrated, slow, crash*) are found.

In [ ]:
risk_keywords = ['cancel', 'pricing', 'billing', 'frustrated', 'unresolved', 'slow', 'crash', 'broken']
keyword_clause = " + ".join([f"CASE WHEN LOWER(sentence) LIKE '%{kw}%' THEN 1 ELSE 0 END" for kw in risk_keywords])

customer_risk_query = f"""
    WITH transcript_risk AS (
        SELECT 
            meeting_id,
            SUM({keyword_clause}) as risk_keyword_count,
            SUM(CASE WHEN sentiment_type = 'negative' THEN 1 ELSE 0 END) as negative_sentences,
            COUNT(*) as total_sentences
        FROM transcript_segments
        GROUP BY meeting_id
    ),
    competitor_risk AS (
        SELECT 
            meeting_id,
            COUNT(DISTINCT entity_name) as competitor_count
        FROM entities
        WHERE entity_type = 'Competitor'
        GROUP BY meeting_id
    )
    SELECT 
        m.title,
        m.overall_sentiment,
        m.sentiment_score,
        COALESCE(tr.risk_keyword_count, 0) as risk_keywords,
        COALESCE(tr.negative_sentences, 0) as negative_sentences,
        COALESCE(tr.total_sentences, 1) as total_sentences,
        COALESCE(cr.competitor_count, 0) as competitor_count,
        ROUND(
            0.4 * (100.0 * (5.0 - m.sentiment_score) / 4.0) + 
            0.3 * (100.0 * COALESCE(tr.negative_sentences, 0)::FLOAT / COALESCE(tr.total_sentences, 1)) +
            (CASE WHEN COALESCE(cr.competitor_count, 0) > 0 THEN 20.0 ELSE 0.0 END) +
            (CASE WHEN COALESCE(tr.risk_keyword_count, 0) > 0 THEN 10.0 ELSE 0.0 END),
            1
        ) as composite_risk_score
    FROM meetings m
    LEFT JOIN transcript_risk tr ON m.meeting_id = tr.meeting_id
    LEFT JOIN competitor_risk cr ON m.meeting_id = cr.meeting_id
    WHERE m.call_type != 'Internal'
    ORDER BY composite_risk_score DESC
"""

df_risk = conn.execute(customer_risk_query).df()
print("Top 10 Customer Engagements sorted by Risk Score:")
print(df_risk.head(10)[['title', 'sentiment_score', 'competitor_count', 'risk_keywords', 'composite_risk_score']])

# Bucket scores into tiers
df_risk['risk_tier'] = pd.cut(df_risk['composite_risk_score'], bins=[0, 30, 60, 100], labels=['Low', 'Medium', 'High'])
print("\nRisk Tier Distributions:")
print(df_risk['risk_tier'].value_counts())

# Visualise
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_risk, x='sentiment_score', y='composite_risk_score', 
    hue='risk_tier', palette={'Low': 'green', 'Medium': 'orange', 'High': 'red'}, 
    size='risk_keywords', sizes=(40, 200), alpha=0.8
)
plt.axhline(60, color='red', linestyle='--', alpha=0.4, label='High Churn Risk Threshold')
plt.title("Customer Churn Risk Scoring Map")
plt.xlabel("Overall Sentiment Score (1.0 to 5.0)")
plt.ylabel("Composite Risk Score (0 to 100)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 6. Product & Competitor Mention Mapping

Let's extract frequencies of competitor mentions and product focus.

In [ ]:
df_entities = conn.execute("SELECT * FROM entities").df()
competitors = df_entities[df_entities['entity_type'] == 'Competitor']['entity_name'].value_counts()
products = df_entities[df_entities['entity_type'] == 'Product']['entity_name'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x=competitors.values, y=competitors.index, ax=axes[0], palette="Oranges_r")
axes[0].set_title("Competitor Mentions Frequency")
sns.barplot(x=products.values, y=products.index, ax=axes[1], palette="Blues_r")
axes[1].set_title("Product Mentions / Load")
plt.tight_layout()
plt.show()

## 7. Algorithm 3: Knowledge Graph Centrality (Node Degree)

This algorithm computes connection density (degree centrality) for each node inside our knowledge graph (`graph_nodes` & `graph_edges`). Highly connected entities represent critical discussion points or active employees in corporate records.

In [ ]:
degree_centrality_query = """
    SELECT n.label, n.type, COUNT(*) as degree
    FROM graph_nodes n
    JOIN (
        SELECT source as node_id FROM graph_edges
        UNION ALL
        SELECT target as node_id FROM graph_edges
    ) e ON n.id = e.node_id
    GROUP BY n.label, n.type
    ORDER BY degree DESC
"""

df_centrality = conn.execute(degree_centrality_query).df()
print("Knowledge Graph Node Degree Centrality (Top 10):")
print(df_centrality.head(10))

plt.figure(figsize=(12, 6))
sns.barplot(data=df_centrality.head(15), x='degree', y='label', hue='type', dodge=False, palette="viridis")
plt.title("Most Central Entities in the Knowledge Graph Network")
plt.xlabel("Degree Centrality (Edges Count)")
plt.ylabel("Entity Label")
plt.tight_layout()
plt.show()

In [ ]:
# Close analytical database connection
conn.close()
print("DuckDB analytical connection closed cleanly.")